# ð¬ Render SubaruShort no Google Colab (GPU T4 Gratuita)

Este notebook configura o ambiente completo para renderizar o projeto **claude-animated-shorts** usando a GPU NVIDIA T4 do Google Colab (12 GB RAM + GPU).

## Composições disponíveis:
- **SubaruShort** - 720x1280, 30 fps, 1800 frames (~60s) - **NOVO: E se você tivesse o poder do Subaru**
- **BridgeShort** - 720x1280, 30 fps, 2025 frames (~67s) - Referência original

---


## 1️⃣ Verificar GPU e Instalar Dependências do Sistema

In [ ]:
# Verificar se GPU está disponível
!nvidia-smi

In [ ]:
# Instalar Node.js 20, ffmpeg e dependências do Chrome headless
!apt-get update -qq && apt-get install -y -qq \
    nodejs npm \
    ffmpeg \
    chromium-browser \
    libnss3 libatk-bridge2.0-0 libdrm2 libxkbcommon0 \
    libxcomposite1 libxdamage1 libxrandr2 libgbm1 \
    libasound2 libatk1.0-0 libc6 libcairo2 libcups2 \
    libdbus-1-3 libexpat1 libfontconfig1 libgcc1 \
    libglib2.0-0 libgtk-3-0 libnspr4 libnss3 libpango-1.0-0 \
    libpangocairo-1.0-0 libstdc++6 libx11-6 libx11-xcb1 \
    libxcb1 libxext6 libxfixes3 libxi6 libxrender1 \
    libxtst6 ca-certificates fonts-liberation \
    libappindicator3-1 lsb-release xdg-utils wget curl

# Instalar edge-tts para narração em português
!pip install -q edge-tts whisperx scipy

# Verificar versões
!node --version && npm --version && ffmpeg -version | head -1 && chromium-browser --version

## 2️⃣ Clonar Repositório e Instalar Dependências NPM

In [ ]:
# ⚠️ SUBSTITUA PELA URL DO SEU REPOSITÓRIO GITHUB
REPO_URL = "https://github.com/SEU_USUARIO/claude-animated-shorts.git"
PROJECT_DIR = "/content/claude-animated-shorts"

import shutil
import os

# Se já existe, remover
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

# Clonar
!git clone $REPO_URL $PROJECT_DIR

# Entrar no diretório
%cd $PROJECT_DIR

In [ ]:
# Instalar dependências npm com --legacy-peer-deps
!npm install --legacy-peer-deps

## 3️⃣ Configurar Skill `cartoon-3d-short` e Assets Kenney

In [ ]:
# Configurar a skill cartoon-3d-short no ~/.claude/skills/
skill_dir = os.path.expanduser("~/.claude/skills/cartoon-3d-short")
os.makedirs(skill_dir, exist_ok=True)

# Copiar SKILL.md do projeto para o local da skill
!cp $PROJECT_DIR/SKILL.md $skill_dir/SKILL.md

# Verificar
!ls -la $skill_dir/

In [ ]:
# Baixar assets Kenney Nature Kit (necessários para o BridgeShort)
kenney_dir = os.path.join(PROJECT_DIR, "public/bridge/kenney")
os.makedirs(kenney_dir, exist_ok=True)

print("📥 Assets Kenney Nature Kit:")
print("   1. Acesse: https://kenney.nl/assets/nature-kit")
print("   2. Baixe o ZIP")
print("   3. Extraia os .glb para:", kenney_dir)
print("   ")
print("💡 Alternativa: Rode o script create_placeholder.py para placeholders temporários")

# Executar script de placeholders (gera .glb mínimos para testar)
!cd $kenney_dir && python3 create_placeholder.py

# Verificar arquivos
!ls -la $kenney_dir/*.glb | head -5

In [ ]:
# Configurar REMOTION_TMP para disco com espaço (Colab tem ~70GB em /content)
os.environ["REMOTION_TMP"] = "/content/remotion-tmp"
os.environ["TMP"] = "/content/remotion-tmp"
os.environ["TEMP"] = "/content/remotion-tmp"

os.makedirs("/content/remotion-tmp", exist_ok=True)

print(f"REMOTION_TMP = {os.environ['REMOTION_TMP']}")
print(f"Espaço livre: {shutil.disk_usage('/content').free / 1e9:.1f} GB")

## 4️⃣ Gerar Áudio Completo (Narração + Música + SFX)

In [ ]:
# Gerar áudio completo usando o script do projeto
%cd $PROJECT_DIR
!python templates/audio/generate_subaru_audio.py

In [ ]:
# Verificar arquivos de áudio gerados
!ls -lh $PROJECT_DIR/subaru-short/

## 5️⃣ Renderizar Still (Frame Único) - Teste Rápido

In [ ]:
# Renderizar frame 0 do SubaruShort com aceleração GPU (--gl=egl)
%cd $PROJECT_DIR

# Usar --gl=egl para GPU NVIDIA T4 (melhor que angle/swiftshader)
!REMOTION_TMP=/content/remotion-tmp \
  npx remotion still src/entry.tsx SubaruShort \
  --frame=0 \
  --timeout=300000 \
  --gl=egl \
  --chromium-flags='{"args":["--disable-dev-shm-usage","--no-sandbox","--use-gl=egl","--enable-gpu-rasterization"]}' \
  --output=out/SubaruShort_frame0.png

In [ ]:
# Verificar resultado
from PIL import Image
img = Image.open("$PROJECT_DIR/out/SubaruShort_frame0.png")
print(f"Dimensões: {img.size}")
print(f"Modo: {img.mode}")
img

## 6️⃣ Renderizar Vídeo Completo SubaruShort 2K (GPU Acelerado) ⭐ PRINCIPAL

In [ ]:
# Renderizar SubaruShort completo com GPU NVIDIA T4
%cd $PROJECT_DIR

!REMOTION_TMP=/content/remotion-tmp \
  npx remotion render src/entry.tsx SubaruShort \
  --concurrency=1 \
  --timeout=600000 \
  --gl=egl \
  --chromium-flags='{"args":["--disable-dev-shm-usage","--no-sandbox","--use-gl=egl","--enable-gpu-rasterization","--enable-zero-copy"]}' \
  --codec=h264 \
  --crf=16 \
  --scale=1.3333 \
  out/SubaruShort_2K.mp4

## 7️⃣ Renderizar BridgeShort 2K (Alternativa - Referência Original)

In [ ]:
# Renderizar BridgeShort completo (comentado por padrão - descomente se quiser testar)
%cd $PROJECT_DIR

# !REMOTION_TMP=/content/remotion-tmp \
#   npx remotion render src/entry.tsx BridgeShort \
#   --concurrency=1 \
#   --timeout=600000 \
#   --gl=egl \
#   --chromium-flags='{"args":["--disable-dev-shm-usage","--no-sandbox","--use-gl=egl","--enable-gpu-rasterization","--enable-zero-copy"]}' \
#   --codec=h264 \
#   --crf=16 \
#   --scale=1.3333 \
#   out/BridgeShort_2K.mp4

## 8️⃣ Baixar Vídeo para o Chromebook

In [ ]:
# Verificar arquivos gerados
!ls -lh $PROJECT_DIR/out/*.mp4

In [ ]:
# Download automático no Colab
from google.colab import files
import glob

mp4_files = glob.glob("$PROJECT_DIR/out/*.mp4")
for f in mp4_files:
    print(f"Baixando: {os.path.basename(f)} ({os.path.getsize(f)/1e6:.1f} MB)")
    files.download(f)

## 9️⃣ Limpeza (Opcional - Libere Espaço)

In [ ]:
# Limpar temp do Remotion (bundle webpack ~1GB cada)
!rm -rf /content/remotion-tmp/remotion-webpack-bundle-*
!rm -rf /content/remotion-tmp/puppeteer_dev_chrome_profile-*

# Verificar espaço liberado
print(f"Espaço livre: {shutil.disk_usage('/content').free / 1e9:.1f} GB")

---

## 📋 Resumo dos Comandos Principais

```bash
# Still (teste rápido) - SubaruShort
npx remotion still src/entry.tsx SubaruShort --frame=0 --timeout=300000 --gl=egl

# Vídeo completo 2K (1440x2560) - SubaruShort ⭐
d83d?
npx remotion render src/entry.tsx SubaruShort \
  --concurrency=1 --timeout=600000 --gl=egl \
  --codec=h264 --crf=16 --scale=1.3333 \
  out/SubaruShort_2K.mp4

# Vídeo completo 2K - BridgeShort (referência)
npx remotion render src/entry.tsx BridgeShort \
  --concurrency=1 --timeout=600000 --gl=egl \
  --codec=h264 --crf=16 --scale=1.3333 \
  out/BridgeShort_2K.mp4
```

## ⚡ Dicas de Performance no Colab
- **Use `--gl=egl`** para GPU NVIDIA (muito mais rápido que `--gl=angle` ou `swiftshader`)
- **`--concurrency=1`** evita OOM na GPU T4 (16GB VRAM)
- **`--scale=1.3333`** renderiza em 2K (1440x2560) a partir do 720x1280 base
- **`REMOTION_TMP=/content/remotion-tmp`** usa o disco do Colab (70GB) em vez de /tmp
- **Limpe `remotion-webpack-bundle-*`** após cada render para não encher o disco
- **Gere áudio ANTES** do render de vídeo (célula 4)

## 🔗 Como Usar Este Notebook

1. **Abra no Colab:** `https://colab.research.google.com/github/SEU_USUARIO/claude-animated-shorts/blob/main/render_colab.ipynb`
2. **Ou faça upload:** No Colab, `Arquivo` → `Fazer upload do notebook` → selecione `render_colab.ipynb`
3. **Ative GPU:** `Ambiente de execução` → `Alterar tipo de ambiente` → `GPU T4`
4. **Edite a linha 52 (Célula 2):** Substitua `REPO_URL` pela URL do seu GitHub
5. **Execute todas as células:** `Ambiente de execução` → `Executar tudo`

## 📦 Arquivos do Projeto Incluídos
- `package.json` - Scripts `still:dev`, `render:dev`, `clean:tmp` otimizados para `--gl=egl`
- `scripts/clean-tmp.js` - Limpeza de temp otimizada
- `render_colab.ipynb` - Este notebook
- `SKILL.md` - Skill cartoon-3d-short
- `templates/audio/subaru_script.md` - Roteiro completo em pt-BR
- `templates/audio/generate_subaru_audio.py` - Geração completa de áudio (TTS + música + SFX)
- `public/bridge/kenney/` - Assets (placeholders + script download)
- `src/subaru/` - Composição completa SubaruShort

---

### 📝 Próximos Passos após Render
1. O arquivo `SubaruShort_2K.mp4` será baixado automaticamente para seu Chromebook
2. Verifique se o áudio está sincronizado (o Remotion escreve faixa silenciosa; use ffmpeg para muxar se precisar)
3. Para YouTube Shorts/TikTok/Reels: o vídeo já está em vertical 9:16 (1440x2560)
4. Legendas: o estilo de referência não usa legendas; se quiser, adicione na edição final